# 04 — Model Comparison: CNN vs BERT vs XLNet

Loads `outputs/*_metrics.json` written by notebooks 01–03 and compares the models on accuracy, precision, recall, F1-score, F1-Fake, ROC-AUC, the confusion matrices and per-category performance. Run notebooks 01–03 first.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv

# True  = train on a LENGTH-BALANCED set (equal fake/genuine in every review-length band), so review length
#         alone cannot decide the label; models and results go to outputs_lengthbal/
# False = original setup (full imbalanced training set); results in outputs/
BALANCE_BY_LENGTH = True
OUT_DIR     = f'{PROJECT_DIR}/outputs_lengthbal' if BALANCE_BY_LENGTH else f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR), '->', OUT_DIR)

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

MODELS = ['CNN', 'BERT', 'XLNet']
M = {m: json.load(open(f'{OUT_DIR}/{m.lower()}_metrics.json')) for m in MODELS if os.path.exists(f'{OUT_DIR}/{m.lower()}_metrics.json')}
cols = ['accuracy', 'precision', 'recall', 'f1_fake', 'f1_macro', 'roc_auc', 'threshold', 'train_size', 'train_time_s', 'infer_time_s']
table = pd.DataFrame({m: {c: M[m].get(c) for c in cols} for m in M}).T
table.to_csv(f'{OUT_DIR}/model_comparison.csv')
table.style.format(precision=4).highlight_max(subset=['accuracy', 'precision', 'recall', 'f1_fake', 'f1_macro', 'roc_auc'], color='#c6efce')

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_fake', 'roc_auc']
ax = table[metrics].astype(float).plot.bar(figsize=(10, 4.5), rot=0, width=0.8)
ax.set_ylim(0, 1); ax.set_title('CNN vs BERT vs XLNet (test set)'); ax.legend(ncol=5, loc='lower center', bbox_to_anchor=(0.5, -0.3))
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_bar.png', dpi=150); plt.show()

In [ ]:
test_y = pd.read_csv(f'{DATA_DIR}/test.csv').dropna(subset=['review_text', 'label']).label.astype(int).values
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for m in M:
    p = np.load(f'{OUT_DIR}/{m.lower()}_test_probs.npy')
    fpr, tpr, _ = roc_curve(test_y, p); ax[0].plot(fpr, tpr, label=f"{m} (AUC={M[m]['roc_auc']:.3f})")
    pr, rc, _ = precision_recall_curve(test_y, p); ax[1].plot(rc, pr, label=m)
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set(title='ROC curve', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].set(title='Precision–Recall (Fake class)', xlabel='Recall', ylabel='Precision'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_roc_pr.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(1, len(M), figsize=(4.2 * len(M), 3.6))
for a, m in zip(np.atleast_1d(ax), M):
    sns.heatmap(np.array(M[m]['confusion_matrix']), annot=True, fmt='d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['Genuine', 'Fake'], yticklabels=['Genuine', 'Fake'])
    a.set(title=m, xlabel='Predicted', ylabel='Actual')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/comparison_confusion.png', dpi=150); plt.show()

In [ ]:
# Per-category F1-Fake (paper Table 6 layout)
per_cat = pd.DataFrame({m: {c: v['f1_fake'] for c, v in M[m]['per_category'].items()} for m in M})
per_cat.to_csv(f'{OUT_DIR}/comparison_per_category_f1.csv')
per_cat.plot.barh(figsize=(9, 4), title='F1-Fake by product category'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/comparison_per_category.png', dpi=150); plt.show()
per_cat.round(4)

In [ ]:
best = table['f1_fake'].astype(float).idxmax()
print(f'Best model by F1-Fake: {best}  (F1-Fake={table.loc[best, "f1_fake"]:.4f}, ROC-AUC={table.loc[best, "roc_auc"]:.4f})')

## Does the model just use review length? Original vs length-balanced training

The original models were trained on the raw data, where long reviews are far more often fake. The length-balanced
models (`BALANCE_BY_LENGTH = True`) were trained with equal fake/genuine reviews in every length band.

- **Full test**: the real test distribution (125k reviews).
- **Length-balanced test**: equal fake/genuine in every length band, so length alone scores ~0.5 ROC-AUC here.
  This is the fairest measure of whether a model reads the text rather than counting words.
- **Short fakes caught**: recall on fake reviews of ≤ 20 words. **Long genuine flagged**: share of genuine reviews of > 100 words wrongly called fake.

In [ ]:
# Review-length bands (words). In the raw data the fake share rises from 1.4% (<=5 words) to 65% (>500 words),
# so length alone scores ROC-AUC 0.85. length_balanced() keeps equal fake and genuine reviews in every band.
LEN_BINS = [0, 5, 10, 20, 40, 70, 100, 150, 250, 500, 10**6]

def length_balanced(df, seed=42):
    bands = pd.cut(df['review_length'], LEN_BINS)
    parts = []
    for _, g in df.groupby(bands, observed=True):
        n = min((g.label == 1).sum(), (g.label == 0).sum())
        parts += [g[g.label == 1].sample(n, random_state=seed), g[g.label == 0].sample(n, random_state=seed)]
    return pd.concat(parts).sample(frac=1, random_state=seed)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
te = pd.read_csv(f'{DATA_DIR}/test.csv').dropna(subset=['review_text', 'label']).reset_index(drop=True)
y, L = te.label.astype(int).values, te.review_length.values
bal = length_balanced(te.assign(label=y)).index.values          # rows of the length-balanced test subset

def row(p, th):
    pred = (p >= th).astype(int)
    return {'acc (full)': accuracy_score(y, pred), 'F1-fake (full)': f1_score(y, pred), 'AUC (full)': roc_auc_score(y, p),
            'acc (len-bal)': accuracy_score(y[bal], pred[bal]), 'F1-fake (len-bal)': f1_score(y[bal], pred[bal]),
            'AUC (len-bal)': roc_auc_score(y[bal], p[bal]),
            'short fakes caught': pred[(L <= 20) & (y == 1)].mean(), 'long genuine flagged': pred[(L > 100) & (y == 0)].mean()}

rows = {'Length only (baseline)': {'AUC (full)': roc_auc_score(y, L), 'AUC (len-bal)': roc_auc_score(y[bal], L[bal])}}
for tag, d in [('original', f'{PROJECT_DIR}/outputs'), ('length-balanced', f'{PROJECT_DIR}/outputs_lengthbal')]:
    for m in ['CNN', 'BERT', 'XLNet']:
        f = f'{d}/{m.lower()}_metrics.json'
        if os.path.exists(f):
            rows[f'{m} ({tag})'] = row(np.load(f'{d}/{m.lower()}_test_probs.npy'), json.load(open(f))['threshold'])
lb = pd.DataFrame(rows).T
lb.to_csv(f'{OUT_DIR}/length_bias_comparison.csv')
print('length-balanced test size:', len(bal))
lb.style.format(precision=3, na_rep='–')

In [ ]:
cols = ['AUC (len-bal)', 'short fakes caught', 'long genuine flagged']
ax = lb.drop(index='Length only (baseline)')[cols].plot.bar(figsize=(11, 4.5), rot=20, width=0.8)
ax.set_ylim(0, 1); ax.set_title('Length bias: original vs length-balanced training'); ax.legend(loc='upper right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/length_bias_comparison.png', dpi=150); plt.show()